**SCAN dataset**

[data available here](https://github.com/brendenlake/SCAN)

---



In [33]:
import os
import re
import math
import requests

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


***downloading SCAN from GitHub***

In [34]:
# =========================
# downloading SCAN from GitHub
# =========================
import os
import requests

def download_scan_addprim(data_dir="data/addprim"):
    """
    Download SCAN add-primitive splits (JUMP + TURN LEFT)
    exactly like in the original notebook.
    """
    base_url = "https://raw.githubusercontent.com/brendenlake/SCAN/master/add_prim_split"

    files = [
        "tasks_train_addprim_jump.txt",
        "tasks_test_addprim_jump.txt",
        "tasks_train_addprim_turn_left.txt",
        "tasks_test_addprim_turn_left.txt",
    ]

    os.makedirs(data_dir, exist_ok=True)

    for fname in files:
        out_path = os.path.join(data_dir, fname)
        if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
            continue

        print(f"Downloading {fname}...")
        url = f"{base_url}/{fname}"
        r = requests.get(url)
        r.raise_for_status()

        with open(out_path, "wb") as f:
            f.write(r.content)

    print("SCAN add-primitive splits ready.")
    return data_dir



***loading SCAN data***

In [35]:
# =========================
# loading SCAN data
# =========================
import re

def load_scan_split(path):
    """
    Load SCAN tasks file.
    Each line: 'IN: ... OUT: ...'
    Returns: list of tokenized inputs, list of tokenized outputs.
    """
    inputs = []
    outputs = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            inp = re.findall(r"IN: (.*) OUT:", line)[0]
            out = line.split("OUT: ")[1]

            inputs.append(inp.split())
            outputs.append(out.split())

    return inputs, outputs


In [36]:
# =========================
# Load both Experiment 3a and 3b
# =========================

data_dir = download_scan_addprim()

# ---- JUMP (Experiment 3a)
jump_train_path = os.path.join(data_dir, "tasks_train_addprim_jump.txt")
jump_test_path  = os.path.join(data_dir, "tasks_test_addprim_jump.txt")

jump_train_inp, jump_train_out = load_scan_split(jump_train_path)
jump_test_inp,  jump_test_out  = load_scan_split(jump_test_path)

print(f"[JUMP] Train samples: {len(jump_train_inp)}")
print(f"[JUMP] Test samples:  {len(jump_test_inp)}")

# ---- TURN LEFT (Experiment 3b)
turn_train_path = os.path.join(data_dir, "tasks_train_addprim_turn_left.txt")
turn_test_path  = os.path.join(data_dir, "tasks_test_addprim_turn_left.txt")

turn_train_inp, turn_train_out = load_scan_split(turn_train_path)
turn_test_inp,  turn_test_out  = load_scan_split(turn_test_path)

print(f"[TURN LEFT] Train samples: {len(turn_train_inp)}")
print(f"[TURN LEFT] Test samples:  {len(turn_test_inp)}")

SCAN add-primitive splits ready.
[JUMP] Train samples: 14670
[JUMP] Test samples:  7706
[TURN LEFT] Train samples: 21890
[TURN LEFT] Test samples:  1208


***vocab builder***

In [37]:
PAD = 0
BOS = 1
EOS = 2

def build_vocab(seqs):
    """
    Build mapping token -> id, including PAD/BOS/EOS.
    """
    vocab = {"<PAD>": PAD, "<BOS>": BOS, "<EOS>": EOS}
    idx = 3
    for seq in seqs:
        for tok in seq:
            if tok not in vocab:
                vocab[tok] = idx
                idx += 1
    return vocab

src_vocab = build_vocab(jump_train_inp)
tgt_vocab = build_vocab(jump_train_out)

src_ivocab = {v: k for k, v in src_vocab.items()}
tgt_ivocab = {v: k for k, v in tgt_vocab.items()}

print("Source vocab size:", len(src_vocab))
print("Target vocab size:", len(tgt_vocab))

Source vocab size: 16
Target vocab size: 9


***dataset & dataloader***

In [38]:
# ==============================
# dataset & dataloader
# ==============================

from torch.utils.data import Dataset, DataLoader, RandomSampler


# ---- helpers ----

def encode(seq, vocab):
    return [vocab[t] for t in seq]

def make_decoder_inputs(seq_ids):
    """
    seq_ids: [y1, y2, ..., yN]
    return:
      tgt_in  = [BOS, y1, ..., yN]
      tgt_out = [y1, ..., yN, EOS]
    """
    return [BOS] + seq_ids, seq_ids + [EOS]


# ---- Dataset ----

class ScanDataset(Dataset):
    def __init__(self, inputs, outputs, src_vocab, tgt_vocab):
        self.inputs = inputs
        self.outputs = outputs
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        src_ids = encode(self.inputs[idx], self.src_vocab)
        tgt_ids = encode(self.outputs[idx], self.tgt_vocab)
        tgt_in, tgt_out = make_decoder_inputs(tgt_ids)

        return (
            torch.tensor(src_ids, dtype=torch.long),
            torch.tensor(tgt_in, dtype=torch.long),
            torch.tensor(tgt_out, dtype=torch.long),
        )


# ---- collate_fn ----

def collate_fn(batch):
    srcs, tgt_ins, tgt_outs = zip(*batch)
    B = len(batch)

    max_src = max(len(s) for s in srcs)
    max_tgt = max(len(t) for t in tgt_ins)

    src_batch = torch.full((B, max_src), PAD, dtype=torch.long)
    tgt_in_batch = torch.full((B, max_tgt), PAD, dtype=torch.long)
    tgt_out_batch = torch.full((B, max_tgt), PAD, dtype=torch.long)

    for i in range(B):
        src_batch[i, :len(srcs[i])] = srcs[i]
        tgt_in_batch[i, :len(tgt_ins[i])] = tgt_ins[i]
        tgt_out_batch[i, :len(tgt_outs[i])] = tgt_outs[i]

    return src_batch, tgt_in_batch, tgt_out_batch


# ==============================
# SELECT EXPERIMENT
# ==============================

EXPERIMENT = "jump"   # "jump" or "turn_left"

if EXPERIMENT == "jump":
    train_inp, train_out = jump_train_inp, jump_train_out
    test_inp,  test_out  = jump_test_inp,  jump_test_out

elif EXPERIMENT == "turn_left":
    train_inp, train_out = turn_train_inp, turn_train_out
    test_inp,  test_out  = turn_test_inp,  turn_test_out

else:
    raise ValueError("Unknown experiment")


# ==============================
# Build datasets & loaders
# ==============================

BATCH_SIZE = 16

train_ds = ScanDataset(train_inp, train_out, src_vocab, tgt_vocab)
test_ds  = ScanDataset(test_inp,  test_out,  src_vocab, tgt_vocab)

train_dl = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

test_dl = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

# TRANSFORMER

In [39]:
class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, num_heads):
        super().__init__()
        # TODO
        assert emb_dim % num_heads == 0, "emb_dim must be divisible by num_heads"

        # Might look redundant, because: self.num_heads * self.head_dim == emb_dim

        self.emb_dim = emb_dim
        self.num_heads = num_heads
        self.head_dim = emb_dim // num_heads

        self.linear_value = nn.Linear(self.emb_dim, self.num_heads * self.head_dim)
        self.linear_key = nn.Linear(self.emb_dim, self.num_heads * self.head_dim)
        self.linear_query = nn.Linear(self.emb_dim, self.num_heads * self.head_dim)

        self.linear_output = nn.Linear(self.num_heads * self.head_dim, self.emb_dim)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        q_len = query.size(1)
        k_len = key.size(1)
        v_len = value.size(1)
        emb_dim = query.size(2)

        assert emb_dim == self.emb_dim, (
            f"Expected emb_dim={self.emb_dim}, but got {emb_dim}"
        )

        # Reminder: PyTorch tensors are backed by
        # 1 flat, contiguous array in memory
        # + shape metadata
        # + stride metadat
        # `view` creates a new interpretation of the same memory (no data copied),
        # giving it a different logical structure.

        # 1. Linear projections
        Q = self.linear_query(query)
        K = self.linear_key(key)
        V = self.linear_value(value)

        # 2. Split embedding dimension into multiple heads
        # Change the "view" of the data (no computation, only reshaping):
        # [batch, seq_len, emb_dim]
        # -> split emb_dim into (num_heads × head_dim)
        # -> [batch, seq_len, num_heads, head_dim]
        # This is valid because emb_dim == num_heads * head_dim
        # Example: [2, 4, 8] with 2 heads -> [2, 4, 2, 4]
        Q = Q.view(batch_size, q_len, self.num_heads, self.head_dim)
        K = K.view(batch_size, k_len, self.num_heads, self.head_dim)
        V = V.view(batch_size, v_len, self.num_heads, self.head_dim)

        # Rearrange tensor dimensions so that attention is computed per head:
        # Move num_heads in front of seq_len, because torch.matmul operates
        # on the last two dimensions and treats all earlier ones as batch dimensions.
        # [batch, seq_len, num_heads, head_dim]
        # -> [batch, num_heads, seq_len, head_dim]
        Q = Q.permute(0, 2, 1, 3)
        K = K.permute(0, 2, 1, 3)
        V = V.permute(0, 2, 1, 3)

        # 3. Scaled dot-product attention
        # Transpose keys so that matrix multiplication works:
        # [batch, heads, seq_len, head_dim]
        # -> [batch, heads, head_dim, seq_len]
        # This allows each query vector to dot-product with all key vectors
        K_t = K.transpose(-2, -1)

        key_out = torch.matmul(Q, K_t)
        key_out = key_out / math.sqrt(self.head_dim)

        # 4. Mask (optional)
        if mask is not None:
            key_out = key_out.masked_fill(mask == 0, -1e20)

        # 5. Softmax over key dimension (which tokens to attend to)
        attention = torch.softmax(key_out, dim=-1)

        # 6. Weighted sum of values
        out = torch.matmul(attention, V)

        # 7. Combine heads back into a single embedding
        # Move heads back behind sequence dimension:
        # [batch, heads, seq_len, head_dim]
        # -> [batch, seq_len, heads, head_dim]
        out = out.permute(0, 2, 1, 3)

        # Ensure memory is contiguous before reshaping
        out = out.contiguous()

        # Concatenate all heads:
        # [batch, seq_len, heads, head_dim]
        # -> [batch, seq_len, emb_dim]
        # (this is where the "concat" of heads happens)
        out = out.view(batch_size, q_len, self.emb_dim)

        # 8. Final linear projection
        out = self.linear_output(out)

        return out


In [40]:
class TransformerBlock(nn.Module):
    def __init__(self, emb_dim, num_heads, dropout, forward_dim):
        super().__init__()

        self.emb_dim = emb_dim
        self.num_heads = num_heads
        self.dropout = nn.Dropout(dropout)
        self.forward_dim = forward_dim

        self.multihead_attention = MultiHeadAttention(self.emb_dim, self.num_heads)

        self.norm1 = nn.LayerNorm(self.emb_dim, eps=1e-6)
        self.norm2 = nn.LayerNorm(self.emb_dim, eps=1e-6)

        self.ffn = nn.Sequential(
            nn.Linear(self.emb_dim, forward_dim),
            nn.ReLU(),
            nn.Linear(forward_dim, self.emb_dim)
        )

    def forward(self, query, key, value, mask):
        # 1. Multi-head attention
        attn_out = self.multihead_attention(query, key, value, mask)

        # 2. Skip connection + normalization
        query_with_attention = (attn_out + query)
        query_with_attention = self.dropout(query_with_attention)
        query_with_attention = self.norm1(query_with_attention)

        # 3. Feed-forward network
        ffn_out = self.ffn(query_with_attention)

        # 4. Skip connection + normalization
        query_with_ffn_out = (ffn_out + query_with_attention)
        query_with_ffn_out = self.dropout(query_with_ffn_out)
        query_with_ffn_out = self.norm2(query_with_ffn_out)

        block_out = query_with_ffn_out

        return block_out

In [41]:
def get_sinusoid_table(max_len, emb_dim):
    def get_angle(pos, i, emb_dim):
        return pos / 10000 ** ((2 * (i // 2)) / emb_dim)

    sinusoid_table = torch.zeros(max_len, emb_dim)
    for pos in range(max_len):
        for i in range(emb_dim):
            if i % 2 == 0:
                sinusoid_table[pos, i] = math.sin(get_angle(pos, i, emb_dim))
            else:
                sinusoid_table[pos, i] = math.cos(get_angle(pos, i, emb_dim))
    return sinusoid_table

In [42]:
class Encoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        emb_dim,
        num_layers,
        num_heads,
        forward_dim,
        dropout,
        max_len,
    ):
        super().__init__()


        # Token embeddings
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)

        # Positional encodings (sinusoid, frozen)
        pos_table = get_sinusoid_table(max_len + 1, emb_dim)
        self.position_embedding = nn.Embedding.from_pretrained(
            pos_table,
            freeze=True
        )

        # Make a dropout layer
        self.dropout = nn.Dropout(dropout)

        # Transformer blocks
        self.layers = nn.ModuleList(
            [
                TransformerBlock(
                    emb_dim=emb_dim,
                    num_heads=num_heads,
                    dropout=dropout,
                    forward_dim=forward_dim
                )
                for _ in range(num_layers)
            ]
        )

    def forward(self, x, mask):
        batch_size, seq_len = x.shape

        # Create position indices [1..seq_len] for each batch element
        positions = torch.arange(seq_len)                # [seq_len]
        positions = positions.unsqueeze(0)               # add batch dim -> [1, seq_len]
        positions = positions.expand(batch_size, seq_len)  # repeat for batch -> [batch, seq_len]
        positions = positions + 1                         # reserve 0 for [PAD]
        positions = positions.to(x.device)                # move to same device (cpu/gpu) as input

        # Embeddings
        token_emb = self.token_embedding(x)
        pos_emb = self.position_embedding(positions)

        # Sum + dropout
        token_pos_emb = token_emb + pos_emb
        token_pos_emb = self.dropout(token_pos_emb)

        encoder_out = token_pos_emb

        # Transformer blocks
        for layer in self.layers:
            encoder_out = layer(encoder_out, encoder_out, encoder_out, mask)

        return encoder_out

In [43]:
class DecoderBlock(nn.Module):
    def __init__(self, emb_dim, num_heads, forward_dim, dropout):
        super().__init__()

        # Masked self-attention (decoder attends to itself)
        self.multihead_attention = MultiHeadAttention(emb_dim, num_heads)

        # LayerNorm after first skip connection
        self.norm = nn.LayerNorm(emb_dim, eps=1e-6)

        # Cross-attention + FFN (reuse TransformerBlock)
        self.transformer_block = TransformerBlock(
            emb_dim=emb_dim,
            num_heads=num_heads,
            dropout=dropout,
            forward_dim=forward_dim
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, value, key, src_mask, tgt_mask):
        # 1. Masked self-attention (decoder attends to itself)
        multihead_attention_out = self.multihead_attention(x, x, x, tgt_mask)

        # 2. Skip connection + normalization
        x_with_self_attn = multihead_attention_out + x
        x_with_self_attn = self.dropout(x_with_self_attn)
        x_with_self_attn = self.norm(x_with_self_attn)

        # 3. Cross-attention + FFN (encoder-decoder attention)
        out = self.transformer_block(
            x_with_self_attn,
            key,
            value,
            src_mask
        )

        return out

In [44]:
class Decoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        emb_dim,
        num_layers,
        num_heads,
        forward_dim,
        dropout,
        max_len
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(max_len, emb_dim)

        self.dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList(
            [
                DecoderBlock(
                    emb_dim=emb_dim,
                    num_heads=num_heads,
                    forward_dim=forward_dim,
                    dropout=dropout
                )
                for _ in range(num_layers)
            ]
        )

        self.output_layer = nn.Linear(emb_dim, vocab_size)

    def forward(self, x, encoder_out, src_mask, tgt_mask):
        batch_size, seq_len = x.shape

        # Create position indices for decoder tokens
        positions = torch.arange(seq_len)                  # [seq_len]
        positions = positions.unsqueeze(0)                 # [1, seq_len]
        positions = positions.expand(batch_size, seq_len)  # [batch, seq_len]
        positions = positions.to(x.device)                 # move to same device as input

        # Token + positional embeddings
        token_emb = self.token_embedding(x)
        pos_emb = self.position_embedding(positions)

        decoder_out = self.dropout(token_emb + pos_emb)

        # Pass through stacked Decoder blocks
        for layer in self.layers:
            decoder_out = layer(
                decoder_out,
                encoder_out,
                encoder_out,
                src_mask,
                tgt_mask
            )

        # Project to vocabulary size
        logits = self.output_layer(decoder_out)

        return logits


In [45]:
class Transformer(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        src_pad_idx,
        tgt_pad_idx,
        emb_dim=512,
        num_layers=6,
        num_heads=8,
        forward_dim=2048,
        dropout=0.0,
        max_len=128,
    ):
        super().__init__()

        self.src_pad_idx = src_pad_idx
        self.tgt_pad_idx = tgt_pad_idx

        self.encoder = Encoder(
            vocab_size=src_vocab_size,
            emb_dim=emb_dim,
            num_layers=num_layers,
            num_heads=num_heads,
            forward_dim=forward_dim,
            dropout=dropout,
            max_len=max_len,
        )

        self.decoder = Decoder(
            vocab_size=tgt_vocab_size,
            emb_dim=emb_dim,
            num_layers=num_layers,
            num_heads=num_heads,
            forward_dim=forward_dim,
            dropout=dropout,
            max_len=max_len,
        )

    def create_src_mask(self, src):
        device = src.device
        # (batch_size, 1, 1, src_seq_len)
        src_mask = (src != self.src_pad_idx).unsqueeze(1).unsqueeze(2)
        return src_mask.to(device)

    def create_tgt_mask(self, tgt):
        device = tgt.device
        batch_size, tgt_len = tgt.shape
        tgt_mask = (tgt != self.tgt_pad_idx).unsqueeze(1).unsqueeze(2)
        tgt_mask = tgt_mask * torch.tril(torch.ones((tgt_len, tgt_len), device=device)).expand(
            batch_size, 1, tgt_len, tgt_len
        )
        return tgt_mask.to(device)

    def forward(self, src, tgt):
        # Create masks
        src_mask = self.create_src_mask(src)
        tgt_mask = self.create_tgt_mask(tgt)

        # Encode source sequence
        encoder_out = self.encoder(src, src_mask)

        # Decode target sequence
        out = self.decoder(
            tgt,
            encoder_out,
            src_mask,
            tgt_mask
        )

        return out

# **Experiment 3.**

In [46]:
import random
import numpy as np
import torch
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

In [47]:
########################################
# Evaluation utilities (SCAN)
########################################

@torch.no_grad()
def greedy_decode(model, src, max_len=100):
    model.eval()
    B = src.size(0)

    ys = torch.full(
        (B, 1),
        BOS,
        dtype=torch.long,
        device=src.device
    )

    for _ in range(max_len):
        logits = model(src, ys)
        next_token = logits[:, -1].argmax(dim=-1, keepdim=True)
        ys = torch.cat([ys, next_token], dim=1)

        if (next_token == EOS).all():
            break

    return ys


def sequence_accuracy(preds, targets):
    correct = 0
    for p, t in zip(preds, targets):
        p = p.tolist()
        t = t.tolist()

        # drop BOS from prediction if present
        if len(p) > 0 and p[0] == BOS:
            p = p[1:]

        if EOS in p:
            p = p[:p.index(EOS)+1]
        if EOS in t:
            t = t[:t.index(EOS)+1]

        if p == t:
            correct += 1
    return correct / len(targets)


@torch.no_grad()
def evaluate_sequence_accuracy(model, dataloader):
    model.eval()
    total_correct = 0
    total = 0

    for src, _, tgt_out in dataloader:
        src = src.to(DEVICE)
        tgt_out = tgt_out.to(DEVICE)

        preds = greedy_decode(model, src)

        batch_acc = sequence_accuracy(preds, tgt_out)
        total_correct += batch_acc * src.size(0)
        total += src.size(0)

    return total_correct / total

In [48]:
# ==============================
# Device
# ==============================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


In [49]:
# ==============================
# TRAINING SETUP
# ==============================

EMB_DIM = 128
N_LAYERS = 2
N_HEADS = 8
FORWARD_DIM = 256
DROPOUT = 0.15
LEARNING_RATE = 2e-4
GRAD_CLIP = 1.0
EPOCHS = 4 #4 epochs for turn_left

model = Transformer(
    src_vocab_size=len(src_vocab),
    tgt_vocab_size=len(tgt_vocab),
    src_pad_idx=PAD,
    tgt_pad_idx=PAD,
    emb_dim=EMB_DIM,
    num_layers=N_LAYERS,
    num_heads=N_HEADS,
    forward_dim=FORWARD_DIM,
    dropout=DROPOUT,
    max_len=100,
).to(DEVICE)

criterion = torch.nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

In [50]:
# ==============================
# TRAINING LOOP
# ==============================

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    steps = 0

    for src, tgt_in, tgt_out in train_dl:
        src = src.to(DEVICE)
        tgt_in = tgt_in.to(DEVICE)
        tgt_out = tgt_out.to(DEVICE)

        optimizer.zero_grad()

        logits = model(src, tgt_in)

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt_out.reshape(-1)
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        running_loss += loss.item()
        steps += 1

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={running_loss/steps:.4f}"
    )

Epoch 01 | train_loss=0.7277
Epoch 02 | train_loss=0.4375
Epoch 03 | train_loss=0.3709
Epoch 04 | train_loss=0.2888


In [60]:
# ==============================
# Greedy decoding (with optional oracle)
# ==============================
@torch.no_grad()
def greedy_decode(
    model,
    source_sequences,
    use_oracle=False,
    target_output_lengths=None,
    max_length=40,   # ↓ уменьшили для скорости, логика та же
):
    model.eval()
    source_sequences = source_sequences.to(DEVICE)

    batch_size = source_sequences.size(0)

    decoded_sequences = torch.full(
        (batch_size, 1),
        BOS,
        dtype=torch.long,
        device=DEVICE,
    )

    finished = torch.zeros(
        batch_size,
        dtype=torch.bool,
        device=DEVICE,
    )

    for step in range(max_length):
        logits = model(source_sequences, decoded_sequences)
        next_tokens = logits[:, -1].argmax(dim=-1, keepdim=True)

        # === ORACLE CONSTRAINTS (same logic as your classmate) ===
        if use_oracle:
            assert target_output_lengths is not None

            for i in range(batch_size):
                tgt_len = int(target_output_lengths[i])

                # too early → forbid EOS
                if step + 1 < tgt_len and next_tokens[i].item() == EOS:
                    logits_i = logits[i, -1]
                    logits_i[EOS] = float("-inf")
                    next_tokens[i] = logits_i.argmax()

                # at or after target length → force EOS
                if step + 1 >= tgt_len:
                    next_tokens[i] = EOS

        finished |= (next_tokens.squeeze(1) == EOS)
        next_tokens[finished] = EOS

        decoded_sequences = torch.cat(
            [decoded_sequences, next_tokens],
            dim=1,
        )

        if finished.all():
            break

    return decoded_sequences

In [61]:
def sequence_accuracy(predicted_sequences, target_sequences):
    correct = 0
    total = target_sequences.size(0)

    predicted_sequences = predicted_sequences.cpu()
    target_sequences = target_sequences.cpu()

    for predicted, target in zip(predicted_sequences, target_sequences):
        predicted = predicted.tolist()
        target = target.tolist()

        if len(predicted) > 0 and predicted[0] == BOS:
            predicted = predicted[1:]

        if EOS in predicted:
            predicted = predicted[: predicted.index(EOS) + 1]
        if EOS in target:
            target = target[: target.index(EOS) + 1]

        if predicted == target:
            correct += 1

    return correct / total if total > 0 else 0.0

In [62]:
@torch.no_grad()
def evaluate_sequence_accuracy(
    model,
    dataloader,
    use_oracle=False,
    max_length=40,
):
    model.eval()
    total_correct = 0
    total_samples = 0

    for src, _, tgt_out in dataloader:
        src = src.to(DEVICE)
        tgt_out = tgt_out.to(DEVICE)

        if use_oracle:
            target_lengths = (tgt_out != PAD).sum(dim=1)
        else:
            target_lengths = None

        preds = greedy_decode(
            model,
            src,
            use_oracle=use_oracle,
            target_output_lengths=target_lengths,
            max_length=max_length,
        )

        batch_acc = sequence_accuracy(preds, tgt_out)
        total_correct += batch_acc * src.size(0)
        total_samples += src.size(0)

    return total_correct / total_samples if total_samples > 0 else 0.0

In [63]:
def token_accuracy_from_logits(logits, targets):
    preds = logits.argmax(dim=-1)
    mask = targets != PAD
    correct = (preds == targets) & mask
    return correct.sum().item() / mask.sum().item()


@torch.no_grad()
def evaluate_token_level(model, dataloader):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    steps = 0

    for src, tgt_in, tgt_out in dataloader:
        src = src.to(DEVICE)
        tgt_in = tgt_in.to(DEVICE)
        tgt_out = tgt_out.to(DEVICE)

        logits = model(src, tgt_in)
        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt_out.reshape(-1),
        )

        acc = token_accuracy_from_logits(logits, tgt_out)

        total_loss += loss.item()
        total_acc += acc
        steps += 1

    return total_loss / steps, total_acc / steps

In [65]:
#non-oracle
val_loss, val_tok_acc = evaluate_token_level(model, test_dl)
val_seq_acc = evaluate_sequence_accuracy(
    model,
    test_dl,
    use_oracle=False,
)

print("\nFinal evaluation (non-oracle):")
print(f"Token-level accuracy:    {val_tok_acc*100:.2f}%")
print(f"Sequence-level accuracy: {val_seq_acc*100:.2f}%")


Final evaluation (non-oracle):
Token-level accuracy:    66.20%
Sequence-level accuracy: 0.22%


In [66]:
#oracle
val_seq_acc_oracle = evaluate_sequence_accuracy(
    model,
    test_dl,
    use_oracle=True,
)

print("\nOracle evaluation:")
print(f"Sequence-level accuracy (oracle): {val_seq_acc_oracle*100:.2f}%")



Oracle evaluation:
Sequence-level accuracy (oracle): 0.74%


In [52]:
val_loss, val_tok_acc = evaluate_token_level(model, test_dl)
val_seq_acc = evaluate_sequence_accuracy(model, test_dl)

print("\nFinal evaluation:")
print(f"Token-level accuracy:    {val_tok_acc*100:.2f}%")
print(f"Sequence-level accuracy: {val_seq_acc*100:.2f}%")


Final evaluation:
Token-level accuracy:    66.20%
Sequence-level accuracy: 0.22%


In [53]:
# ==============================
# SAVE RESULTS TO SEPARATE FILES
# ==============================

import csv

if EXPERIMENT == "jump":
    filename = "experiment3_jump_results.csv"

elif EXPERIMENT == "turn_left":
    filename = "experiment3_turn_left_results.csv"

else:
    raise ValueError("Unknown experiment")

with open(filename, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "experiment",
        "epochs",
        "token_accuracy",
        "sequence_accuracy"
    ])
    writer.writerow([
        EXPERIMENT,
        EPOCHS,
        val_tok_acc,
        val_seq_acc
    ])

print(f"Results saved to {filename}")

Results saved to experiment3_jump_results.csv


***jump composed***

In [67]:
# ==============================
# Download full SCAN commands (tasks.txt)
# ==============================

import os
import urllib.request

tasks_url = "https://raw.githubusercontent.com/brendenlake/SCAN/master/tasks.txt"
tasks_path = os.path.join(data_dir, "tasks_full.txt")

if not os.path.exists(tasks_path):
    urllib.request.urlretrieve(tasks_url, tasks_path)
    print("Downloaded full SCAN tasks.")
else:
    print("Full SCAN tasks already exist.")

Full SCAN tasks already exist.


In [68]:
# ==============================
# Load full SCAN file (tasks.txt)
# ==============================

def load_scan_full(path):
    inputs = []
    outputs = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            inp, out = line.split(" OUT: ")
            inp = inp.replace("IN: ", "").split()
            out = out.split()
            inputs.append(inp)
            outputs.append(out)
    return inputs, outputs


In [69]:
# ==============================
# Load full SCAN tasks
# ==============================

full_inp, full_out = load_scan_full(tasks_path)

print("Total SCAN commands:", len(full_inp))

Total SCAN commands: 20910


In [70]:
# ---- build pool of composed JUMP commands (from full SCAN, NOT test!) ----
jump_composed_pool = [
    (inp, out)
    for inp, out in zip(full_inp, full_out)
    if "jump" in inp and len(inp) > 1
]

print(f"Composed JUMP pool size: {len(jump_composed_pool)}")

Composed JUMP pool size: 7706


In [58]:
def select_first_k_from_pool(k):
    return (
        [inp for inp, _ in jump_composed_pool[:k]],
        [out for _, out in jump_composed_pool[:k]],
    )

In [72]:
# ==============================
# JUMP CURVE: non-oracle + oracle
# ==============================

ks = [0, 1, 2, 4, 8, 16, 32]

token_accs = []
seq_accs_non_oracle = []
seq_accs_oracle = []

for k in ks:
    print(f"\n=== JUMP with {k} composed commands in training ===")

    # --- training data ---
    extra_inp, extra_out = select_first_k_from_pool(k)

    train_inp_k = jump_train_inp + extra_inp
    train_out_k = jump_train_out + extra_out

    # --- vocab (TRAIN ONLY) ---
    src_vocab_k = build_vocab(train_inp_k)
    tgt_vocab_k = build_vocab(train_out_k)

    # --- datasets ---
    train_ds_k = ScanDataset(train_inp_k, train_out_k, src_vocab_k, tgt_vocab_k)
    test_ds_k  = ScanDataset(jump_test_inp, jump_test_out, src_vocab_k, tgt_vocab_k)

    train_dl_k = DataLoader(
        train_ds_k,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn
    )

    test_dl_k = DataLoader(
        test_ds_k,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn
    )

    # --- model ---
    model_k = Transformer(
        src_vocab_size=len(src_vocab_k),
        tgt_vocab_size=len(tgt_vocab_k),
        src_pad_idx=PAD,
        tgt_pad_idx=PAD,
        emb_dim=EMB_DIM,
        num_layers=N_LAYERS,
        num_heads=N_HEADS,
        forward_dim=FORWARD_DIM,
        dropout=DROPOUT,
        max_len=100,
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model_k.parameters(), lr=LEARNING_RATE)
    criterion = torch.nn.CrossEntropyLoss(ignore_index=PAD)

    # --- training ---
    for _ in range(EPOCHS):
        model_k.train()
        for src, tgt_in, tgt_out in train_dl_k:
            src = src.to(DEVICE)
            tgt_in = tgt_in.to(DEVICE)
            tgt_out = tgt_out.to(DEVICE)

            optimizer.zero_grad()
            logits = model_k(src, tgt_in)

            loss = criterion(
                logits.view(-1, logits.size(-1)),
                tgt_out.view(-1)
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_k.parameters(), GRAD_CLIP)
            optimizer.step()

    # --- evaluation ---
    _, tok_acc = evaluate_token_level(model_k, test_dl_k)

    seq_acc_non_oracle = evaluate_sequence_accuracy(
        model_k,
        test_dl_k,
        use_oracle=False,
    )

    seq_acc_oracle = evaluate_sequence_accuracy(
        model_k,
        test_dl_k,
        use_oracle=True,
    )

    token_accs.append(tok_acc * 100)
    seq_accs_non_oracle.append(seq_acc_non_oracle * 100)
    seq_accs_oracle.append(seq_acc_oracle * 100)

    print(
        f"Token acc: {tok_acc*100:.2f}% | "
        f"Seq acc (non-oracle): {seq_acc_non_oracle*100:.2f}% | "
        f"Seq acc (oracle): {seq_acc_oracle*100:.2f}%"
    )

# ==============================
# Final results
# ==============================
print("\n=== FINAL JUMP CURVE RESULTS ===")
for i, k in enumerate(ks):
    print(
        f"k={k:>2d} | "
        f"Token: {token_accs[i]:6.2f}% | "
        f"Seq (non-oracle): {seq_accs_non_oracle[i]:6.2f}% | "
        f"Seq (oracle): {seq_accs_oracle[i]:6.2f}%"
    )


=== JUMP with 0 composed commands in training ===
Token acc: 69.19% | Seq acc (non-oracle): 0.40% | Seq acc (oracle): 1.22%

=== JUMP with 1 composed commands in training ===
Token acc: 71.49% | Seq acc (non-oracle): 0.30% | Seq acc (oracle): 1.14%

=== JUMP with 2 composed commands in training ===
Token acc: 69.75% | Seq acc (non-oracle): 0.14% | Seq acc (oracle): 0.40%

=== JUMP with 4 composed commands in training ===
Token acc: 71.42% | Seq acc (non-oracle): 0.22% | Seq acc (oracle): 0.77%

=== JUMP with 8 composed commands in training ===
Token acc: 74.25% | Seq acc (non-oracle): 3.31% | Seq acc (oracle): 6.11%

=== JUMP with 16 composed commands in training ===
Token acc: 75.45% | Seq acc (non-oracle): 2.37% | Seq acc (oracle): 7.07%

=== JUMP with 32 composed commands in training ===
Token acc: 90.48% | Seq acc (non-oracle): 23.93% | Seq acc (oracle): 52.47%

=== FINAL JUMP CURVE RESULTS ===
k= 0 | Token:  69.19% | Seq (non-oracle):   0.40% | Seq (oracle):   1.22%
k= 1 | Token: